# Getting started with `istari_fluent`

This notebook is a guided first experience with the Istari Digital Platform from Python. You will connect, upload a spreadsheet, run an extraction job, chain a second job, and trace the lineage of the final output &mdash; all in about **15 minutes**.

We use [`istari_fluent`](../fluent): an opinionated, chainable wrapper over the official [`istari-digital-client`](https://docs.istaridigital.com/developers/SDK/01-setup). The concepts (Systems, Models, Jobs, Products, Resources) are identical to the platform UI and to the [Python Client 201 tutorial](https://docs.istaridigital.com/tutorials/python-client/201) &mdash; `istari_fluent` just packages them behind entity-oriented methods.

### By the end you will know how to

- **Connect** to the platform from a notebook using a Personal Access Token.
- Find or create a **System** &mdash; the container for a related set of models and jobs.
- **Register** a file as a Model.
- **Run** an extraction job and **inspect** what it produced.
- **Download** an output and **chain** it into a second job as a source.
- Trace the backward **lineage** of any revision to see how it was created.

### Prerequisites

- An **Istari Digital Platform account** and a **Personal Access Token**. If you don't have an account, follow the [Sign-up Guide](https://docs.istaridigital.com/users/account/sign-up). For token help, see [Personal Access Tokens](https://docs.istaridigital.com/users/user-guide/settings#developer-settings--personal-access-tokens).
- An agent with the **Open Spreadsheet** integration and access to run `@istari:extract`. Your admin can confirm; see [Manage Tool Access](https://docs.istaridigital.com/users/admin-guide/user-management#manage-tool-access-for-a-user).
- Working through [Platform 101](https://docs.istaridigital.com/tutorials/platform/platform-101) first is recommended so the UI concepts (models, jobs, resources, revisions) feel familiar.
- The `Group3-UAS-Requirements.xlsx` sample file next to this notebook (already present in `samples/`).

### 1 &middot; Credentials

Create `samples/.env` (next to this notebook) with the same two variable names the official Python client uses:

```
ISTARI_REGISTRY_URL=https://...paste your platform's registry URL here...
ISTARI_PERSONAL_ACCESS_TOKEN=...paste your token here...
```

You can find both in the platform under **Settings &rarr; Developer Settings**. Treat the token as a secret &mdash; it grants API access as you. Don't commit it, don't paste it into chat, don't screenshot it.

> The connect cell below looks for `samples/.env` first, then falls back to `fluent/.env`.

### 2 &middot; Install dependencies

From the repository root:

```bash
cd fluent
uv sync --extra experiment
```

This creates `fluent/.venv/` containing `istari_fluent`, `jupyter`, and `ipykernel`.

### 3 &middot; Register the venv as a Jupyter kernel

A bare virtualenv is not auto-discovered by VS Code / Cursor. Register it once:

```bash
# still inside fluent/
uv run python -m ipykernel install --user --name istari-fluent --display-name "Python (istari_fluent)"
```

Then reload this notebook's kernel picker (top-right) and select **"Python (istari_fluent)"**.

To remove the kernel later: `jupyter kernelspec uninstall istari-fluent`.

> **A note on `istari_fluent`** &mdash; this is a productivity layer maintained alongside the official SDK. It is not the officially supported client. For production integrations, keep the core [`istari-digital-client`](https://docs.istaridigital.com/developers/SDK/01-setup) as your source of truth; use `istari_fluent` to prototype, explore, and build notebooks faster.

## 1 &middot; Connect and verify

`IstariPlatform.from_env()` reads `ISTARI_REGISTRY_URL` and `ISTARI_PERSONAL_ACCESS_TOKEN` (the same names used by the official Python client) and returns an `IstariPlatform` object.

The token is issued to *you*, so every call the notebook makes acts **on your behalf** &mdash; the same authorization rules you see in the UI apply here. If something is denied by the API, the cause is typically the same as in the UI; see [Sharing and access](https://docs.istaridigital.com/users/user-guide/sharing-and-access).

After connecting we run a quick **readiness check**: one round-trip that confirms the platform is reachable and your token is accepted.

In [2]:
from pathlib import Path

from istari_fluent import IstariPlatform, JobDefinition

# Look for .env next to this notebook (samples/.env), then fall back to fluent/.env.
_candidates = [
    Path.cwd() / ".env",
    Path.cwd().parent / "fluent" / ".env",
]
env_path = next((p for p in _candidates if p.exists()), None)
assert env_path is not None, (
    f"No .env found. Looked in: {[str(p) for p in _candidates]}. "
    "Create samples/.env with ISTARI_REGISTRY_URL and ISTARI_PERSONAL_ACCESS_TOKEN."
)
print(f"Using credentials from: {env_path}")

platform = IstariPlatform.from_env(dotenv_path=str(env_path))

# Readiness check: one round-trip to confirm the platform answers and the token is accepted.
report = platform.client.readiness_check()
print(report)
assert report.healthy, f"Platform reports unhealthy: {report}"

platform

IstariPlatform(url='?')

## 2 &middot; Find or create a System

A **System** groups related Models and their jobs, and gives you a **baseline configuration** that pins which revision of each file is currently authoritative. Think of it as the workspace for a product or subsystem under design.

`get_or_create_system` is an idempotent helper: it looks up by name and creates the System only if nothing matches. Running this cell twice won't create duplicates.

In [ ]:
SYSTEM_NAME = "UAS-Tutorial"

system = platform.get_or_create_system(
    SYSTEM_NAME,
    description="Group 3 UAS tutorial workspace (istari_fluent getting started)",
)
print(system)
print("Baseline configuration:", system.baseline.configuration.name)

## 3 &middot; Register the spreadsheet as a Model

In Istari terminology, registering a file creates a **Model** &mdash; a stable identity for the file with a version history of **revisions**. Each upload or update adds a new revision to the same Model id; the platform's file comparison, provenance, and job inputs all hinge on these revisions.

The `Group3-UAS-Requirements.xlsx` sample is already next to this notebook. We tag the registration with a stable `external_id` so re-running this notebook finds the existing Model instead of creating a duplicate.

In [ ]:
XLSX_PATH = Path.cwd() / "Group3-UAS-Requirements.xlsx"
EXTERNAL_ID = "fluent-tutorial-uas-requirements"

model = platform.find_model(external_id=EXTERNAL_ID)
if model is None:
    model = platform.upload_model(
        XLSX_PATH,
        external_id=EXTERNAL_ID,
        display_name="Group3-UAS-Requirements (tutorial)",
    )
    print("Uploaded new model.")
else:
    print("Reusing existing model.")

print(model)

## 4 &middot; Run the first extraction job

A **Job** is an instruction to the platform to run a specific **function** from a **tool** against a Model revision. Here we run `@istari:extract` with the `open_spreadsheet` tool &mdash; the same function you would select under **Jobs &rarr; Create Job** in the UI, and the same one used by the [Python Client 201 tutorial](https://docs.istaridigital.com/tutorials/python-client/201).

On this spreadsheet the extraction writes four artifacts:

| Artifact | What it contains |
|---|---|
| `named_cells.json` | Values of named ranges (`SubTitle`, `max_weight_value`, ...) |
| `worksheet_data.json` | Full cell data per sheet |
| `workbook.pdf` | Rendered PDF view of the workbook |
| `workbook.html` | Rendered HTML view |

`model.run_job(definition)` submits the job, polls until it reaches a terminal state, and returns the completed `JobView`. It raises if the job fails or the timeout elapses, so you can treat success as "no exception."

> **Tip:** if you want to submit and poll yourself, use `model.submit_job(definition)` and then `.wait()` / `.on_success()` on the returned `JobView`.

In [ ]:
extract = JobDefinition(
    function="@istari:extract",
    tool_name="open_spreadsheet",
)

job1 = model.run_job(extract, timeout=600)
print(f"Job 1: {job1.status}  id={job1.id}")

## 5 &middot; Inspect the products

Every job records the exact artifact revisions it wrote. `job1.get_products()` reads `job.revision.products` and returns a list of `ResourceView` objects **pinned to those revisions**. This matters: if Job 2 later writes a new revision of `named_cells.json`, the pinned view from Job 1 still points to the original bytes &mdash; just like the Resources tab for a specific job in the UI.

Look at the print-out below: every product has a `file_id` **and** a `rev_id`. The file id is the stable identity of the artifact file on the platform; the revision id is the specific version this job produced.

In [ ]:
products_1 = job1.get_products()
print(f"Job 1 wrote {len(products_1)} products:\n")
for p in products_1:
    print(f"  - {p.type:10s}  name={p.name!r:30s}  file={p.file_id}  rev={p.revision_id}")

## 6 &middot; Pick a product and download it

Once we have a pinned `ResourceView`, we can read its bytes straight into memory (`read_bytes()`), decode text (`read_text()`), or save to disk (`download(dir)`). Because the view is pinned, these calls always return the same revision even if new revisions are added later.

Here we take the `named_cells.json` product, load it into a Python dict, and also save a copy next to the notebook for your records &mdash; exactly the flow used by [Python Client 201 &sect; Register and run the first extraction](https://docs.istaridigital.com/tutorials/python-client/201#register-the-model-and-run-the-first-extraction).

In [ ]:
import json

named_cells = job1.find_product(filename="named_cells.json")
assert named_cells is not None, "named_cells.json not produced"

data = json.loads(named_cells.read_text())
print(f"named_cells.json has {len(data)} named ranges")
print("First 3 keys:", list(data)[:3])

download_path = named_cells.download(Path.cwd() / "outputs")
print(f"Downloaded to: {download_path}")

## 7 &middot; Chain a second job

Real workflows rarely stop at one job. The platform's **provenance graph** lets you attach any revision as a declared input of a new job; downstream you can always trace back to the exact bytes and the user who produced them.

We'll run the same extraction a second time, and this time feed one of Job 1's products in as an explicit **source** via `as_source()`. The platform stores the link in the lineage graph; later, `get_lineage()` will walk it for us.

`as_source()` uses the product's `revision_id` directly (already present on the `Product` record from Step 5), so this does **not** cost an extra round-trip.

In [ ]:
named_cells_source = named_cells.as_source(relationship_identifier="input")
print("Attaching source:", named_cells_source)

job2 = model.run_job(
    extract,
    sources=[named_cells_source],
    timeout=600,
)
print(f"Job 2: {job2.status}  id={job2.id}")

products_2 = job2.get_products()
print(f"\nJob 2 wrote {len(products_2)} products:")
for p in products_2:
    print(f"  - {p.type:10s}  name={p.name!r:30s}  rev={p.revision_id}")

## 8 &middot; Trace the lineage

The payoff of uploading, running jobs, and chaining sources is that **every revision knows how it got there**. `get_lineage()` walks backward from any revision and classifies each step:

| Step | Meaning |
|---|---|
| `upload` | A fresh file &mdash; no sources, the root of a chain |
| `job_run` | Produced by a Job |
| `promotion` | A Model promoted from another revision (tag `promoted_from`) |
| `derived` | Any other derivation |

Below we pick one of Job 2's products and print the full chain back to the original upload.

In [ ]:
final_output = job2.find_product(filename="named_cells.json")
assert final_output is not None

tree = final_output.get_lineage(max_depth=6)
print("Lineage for Job 2's named_cells.json:\n")
tree.print_tree()

You can also iterate the tree flat, e.g. to find the originating upload or count how many job runs sit in the chain.

In [ ]:
from collections import Counter

steps = Counter(node.step for node in tree.walk())
print("Step counts:", dict(steps))

uploads = [n for n in tree.walk() if n.step == "upload"]
for u in uploads:
    print(f"Origin upload: {u.resource_type} {u.label!r} (rev={u.revision_id})")

## Verify in the UI

The notebook and the platform UI are not parallel demos; they act on the **same** objects. Sign in to the same platform you used for your token and cross-check:

1. **Systems** &mdash; Your `UAS-Tutorial` System should appear, with the baseline configuration shown above.
2. **Files / Models** &mdash; Open `Group3-UAS-Requirements` and match the Model id and revision ids to the printout from Step 3.
3. **Jobs / Activity** &mdash; Two **Completed** extractions for `open_spreadsheet` / `@istari:extract`. The second one lists `named_cells.json` (from Job 1) under its **Sources**.
4. **Resources** &mdash; On each job, confirm the produced artifacts (`named_cells.json`, `worksheet_data.json`, `workbook.pdf`, `workbook.html`). Revision ids match the `rev=` values printed in Steps 5 and 7.
5. **Lineage** &mdash; The platform's lineage view on Job 2's `named_cells.json` shows the same chain that `get_lineage().print_tree()` rendered above.

If any check fails, compare the ids &mdash; in almost every case the script and the UI agree, because they read from the same underlying records.

## What you learned

| Concept | Platform UI | `istari_fluent` |
|---|---|---|
| Connect | Sign in | `IstariPlatform.from_env()` |
| Organize work in a System | Create System | `platform.get_or_create_system(name)` |
| Register a file | Drag and drop | `platform.upload_model(path, external_id=...)` |
| Run an extraction | Create Job | `model.run_job(JobDefinition(...))` |
| See what a job produced | Resources tab on a Job | `job.get_products()` / `job.find_product(filename=...)` |
| Download a resource | Click Download | `product.read_bytes()` / `product.download(dir)` |
| Chain an artifact into a new job | Select as Source in Create Job | `product.as_source()` passed to `run_job(sources=[...])` |
| Trace provenance | Lineage view | `view.get_lineage().print_tree()` |

Every call the notebook made used your Personal Access Token, so everything above happened as *you* &mdash; the same authorization that governs the UI governs the API.

## What's next

- **Parameter studies** &mdash; see [SDK &sect; Parameter study](https://docs.istaridigital.com/developers/SDK/02-patterns/01-workflows/02-parameter-study).
- **Sequential pipelines** &mdash; see [SDK &sect; Sequential pipeline](https://docs.istaridigital.com/developers/SDK/02-patterns/01-workflows/01-sequential-pipeline).
- **Promote products to reusable Models** &mdash; `ResourceView.promote()` creates a standalone Model out of a product, with `promoted_from` recorded in its lineage.
- **Run jobs directly on artifacts** &mdash; `artifact.run_job(...)` auto-promotes the underlying revision to a Model before submitting, so you can keep chaining without an explicit promotion step.
- **Evolve system baselines** &mdash; use `TrackedFileSet` and `ConfigurationView.add_file(...).save()` to roll forward a system's baseline through configuration versions.
- **Full API reference** &mdash; [Python client reference](https://docs.istaridigital.com/developers/SDK/api_reference/01-client).